# Data Cleaning & Quality Audit — Trilingual BANKING77 Bake-off

Interactive review of every data-quality issue that would affect the classifier
bake-off, across all five language/script folders (`english`, `sinhala`, `singlish`,
`tamil`, `tamilish`) x both splits.

This notebook is **non-destructive** — it inspects and exports flagged rows to
`cleaning_report/*.csv` for hand review. It does **not** modify the source datasets.
The audit logic lives in `data_cleaning.py`; this notebook imports its `get_*`
collectors so there is a single source of truth.

Run top-to-bottom, then use the `view_group(...)` helper to eyeball individual
collisions.

In [ ]:
import importlib
import pandas as pd

import data_cleaning as dc
importlib.reload(dc)

# show full ticket text — these are short strings, never truncate for review
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)

data = dc.load()
present = sorted(data.keys())
print('loaded splits:', present)

## Issue summary

One-line count of every issue class. Numbers here match `python data_cleaning.py`.

In [ ]:
mismatches   = dc.get_label_mismatches(data)
untranslated = dc.get_untranslated(data)
conflicts    = dc.get_conflicting_duplicates(data)
exact_dups   = dc.get_exact_duplicates(data)
leakage      = dc.get_leakage(data)

summary = pd.DataFrame([
    {'issue': 'cross-language label mismatches', 'rows': len(mismatches)},
    {'issue': 'untranslated rows (text == text_en)', 'rows': len(untranslated)},
    {'issue': 'conflicting-label collisions (member rows)', 'rows': len(conflicts)},
    {'issue': '  -> distinct collision groups',
     'rows': int(conflicts.groupby(['lang','split'])['group'].nunique().sum()) if len(conflicts) else 0},
    {'issue': 'exact duplicate rows (same text + labels)', 'rows': len(exact_dups)},
    {'issue': 'train/test leakage (test rows)', 'rows': len(leakage)},
]).set_index('issue')
summary

## 1. Conflicting-label duplicates  ⚠️ main concern

One `text` string that carries **more than one** distinct
`(category, sentiment, priority)` tuple within a split. These arise when two
*different* English tickets translate to the *same* target-language text but keep
their different English-derived labels — so the model sees the identical input with
contradictory targets. This is the real label-noise to decide on.

In [ ]:
# counts per language/split
if len(conflicts):
    display(conflicts.groupby(['lang','split'])['group'].nunique()
            .rename('collision_groups').to_frame())
else:
    print('no conflicting-label duplicates')

In [ ]:
def view_group(lang, split, group):
    """Show all member rows of one collision, side by side."""
    g = conflicts[(conflicts.lang == lang) & (conflicts.split == split)
                  & (conflicts.group == group)]
    return g[['id', 'text', 'text_en', 'category', 'sentiment', 'priority']]

# example: first collision in sinhala train
view_group('sinhala', 'train', 0)

In [ ]:
# browse every collision in one language/split as a flat, grouped table
LANG, SPLIT = 'sinhala', 'train'
conflicts[(conflicts.lang == LANG) & (conflicts.split == SPLIT)]\
    .sort_values(['group', 'id'])[['group', 'id', 'text', 'text_en',
                                    'category', 'sentiment', 'priority']]

## 2. Train / test leakage

Test rows whose English source ticket (`text_en`) also appears in **train** — a model
that memorised train gets these test rows for free, inflating apparent accuracy.
Because all five languages are id-aligned, dropping these ids from `test` (or `train`)
must be done for every language together.

In [ ]:
leakage

## 3. Untranslated rows

Non-English rows where `text` == `text_en` verbatim — the translation was skipped or
the source was left as-is. Some are legitimately untranslatable (numbers, short
English-only strings); others are genuine gaps. Review each.

In [ ]:
untranslated[['lang', 'split', 'id', 'text_en', 'text',
              'category', 'sentiment', 'priority']]

## 4. Exact duplicates (safe to dedup)

Rows sharing **both** text and full label tuple — pure redundancy, no conflict. Safe
to collapse to one row per (text, label) within a split if desired; harmless if left.

In [ ]:
if len(exact_dups):
    display(exact_dups.groupby(['lang','split']).size().rename('rows').to_frame())
else:
    print('no exact duplicates')

## 5. Class balance & length reference

Distributional context for the bake-off (drives the oversampling decision and the
macro-F1 emphasis). Labels are identical across languages, so English train is
canonical.

In [ ]:
en = data[('english', 'train')]
for task in ['sentiment', 'priority']:
    vc = en[task].value_counts()
    print(f'{task}  (imbalance {vc.max()/vc.min():.1f}x)')
    print((vc / len(en) * 100).round(1).astype(str).add('%').to_string(), '\n')
cat = en['category'].value_counts()
print(f'category: {len(cat)} classes, min={cat.min()} max={cat.max()} median={int(cat.median())}')

## Full text report

Re-run the complete console audit and refresh the `cleaning_report/*.csv` files for
opening in a spreadsheet.

In [ ]:
dc.report(data, do_write=True)

---
### Cleaning decisions to settle (not yet applied)

Nothing above mutates the datasets. The open policy calls, each affecting every
downstream model:

1. **Conflicting-label duplicates** — drop them, keep the majority-label row, or
   leave as-is? (Per-language, since collisions differ by language.)
2. **Train/test leakage** — drop the leaked ids from every language's split, or accept
   ~0.2% overlap?
3. **Exact duplicates** — dedup within split, or keep (affects class weighting)?
4. **Untranslated rows** — re-translate, drop, or accept as legitimately-English?

Cleaning must stay **id-aligned across all five languages** for the pooled
experiments — any row dropped for pooled runs should be dropped consistently.